# ResNet-20 스테이지별 구조적 프루닝 + 지식증류(KD)

CIFAR-10 으로 학습된 ResNet-20 을 채널 단위로 잘라(구조적 프루닝) 계산량을 줄이고,
fine-tuning 과 지식증류(Knowledge Distillation, KD) 로 정확도를 얼마나 회복할 수 있는지 실험합니다.

**질문**

1. ResNet-20 의 세 스테이지(A·B·C) 중 어디가 프루닝에 민감한가? (Part 1)
2. 스테이지마다 다른 비율로 자르면, 전 층 균등 20% 프루닝보다 작으면서 비슷한 정확도를 낼 수 있는가? (Part 2)

**실행 환경**

- **Colab**: 런타임 유형을 **T4 GPU** 로 설정합니다. 셀 1 에서 구글 드라이브 연결을 허용하면
  `내 드라이브/resnet20-stage-pruning-kd/` 아래 `data/` · `results/` · `figures/` 에 저장됩니다.
- **로컬**: 레포 루트에서 `pip install -r requirements.txt` 후, 이 노트북을 `notebooks/` 에 두고 실행합니다.

**데이터 · 가중치 출처**: CIFAR-10 학습 10,000장 · 평가 2,000장 부분집합과 사전학습된 ResNet-20 가중치는
[imguru-mooc/AI_CAMPUS](https://github.com/imguru-mooc/AI_CAMPUS) 저장소에서 실행 시 내려받습니다 (이 레포에는 포함하지 않음).

---

# Part 1 — 스테이지 민감도: shortcut 경로만 잘라 보기

ResNet 은 shortcut(residual) 연결 때문에 같은 스테이지 안의 블록 출력 채널이 하나로 묶여 있습니다.
이 묶음을 **shortcut 경로**라고 부르고, 스테이지마다 하나씩 총 3개입니다.

| 경로 | 범위 | 채널 |
|---|---|---|
| A | stem · layer1 | 16 |
| B | layer2 | 32 |
| C | layer3 (→ fc 입력) | 64 |

튜플 `(A, B, C)` 의 각 자리는 경로 하나이고, `1` = 50% 자르기, `0` = 그대로입니다.
Part 1 은 **fine-tuning 없이** 자르기만 해서 어느 경로가 더 민감한지 비교합니다.

| 셀 | 내용 | 시간 |
|---|---|---|
| 1 | 설치 · 불러오기 | 처음 1~2분 |
| 2 | 평가 데이터 · ResNet-20 불러오기 | 수십 초 |
| 3 | `prune_paths` (경로 튜플로 자르기) | 바로 |
| 4 | 실험 1 (누적) · 실험 2 (단독) | 1분 안팎 |
| 5 | 그래프 (선택) | 바로 |
| 6 | 경로별 비율 스윕 (선택) | 1분 안팎 |

In [ ]:
# ▶ 셀 1 — 설치 · 불러오기
import os, sys, copy, time, urllib.request, importlib.util, subprocess, warnings
# 없는 패키지만 설치 (로컬은 requirements.txt 로 미리 설치 권장)
for mod, pkg in [("torch_pruning", "torch-pruning==1.6.0"), ("cv2", "opencv-python")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
import torch_pruning as tp
from IPython.display import display

warnings.filterwarnings("ignore")
torch.manual_seed(0); np.random.seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
# 폴더 위치
#   Colab  → 구글 드라이브에 저장 (런타임이 끝나도 유지)
#   로컬   → notebooks/ 안에서 실행하면 상위 폴더가 레포 루트
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/resnet20-stage-pruning-kd"
else:
    ROOT = os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else ".")
DATA_DIR, RESULTS_DIR, FIG_DIR = (os.path.join(ROOT, d) for d in ("data", "results", "figures"))
for d in (DATA_DIR, RESULTS_DIR, FIG_DIR): os.makedirs(d, exist_ok=True)
print("레포 루트:", ROOT)
print("PyTorch", torch.__version__, "| torch-pruning", getattr(tp, "__version__", "?"), "| 장치", DEV)

In [ ]:
# ▶ 셀 2 — 평가용 CIFAR-10 2,000장 · 사전학습 ResNet-20
BASE = "https://imguru-mooc.github.io/AI_CAMPUS/QP-day5"
RAW = "https://raw.githubusercontent.com/imguru-mooc/AI_CAMPUS/main/QP-day5"

def fetch(name):
    # data/<name> 이 없으면 원본 저장소에서 내려받는다
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        for url in [f"{BASE}/data/{name}", f"{RAW}/data/{name}"]:
            try:
                urllib.request.urlretrieve(url, path); break
            except Exception as e:
                if os.path.exists(path): os.remove(path)
                print("내려받기 실패:", url, "→", e)
        else:
            raise FileNotFoundError(name)
    return path

MEAN = np.array((0.4914, 0.4822, 0.4465), np.float32)
STD = np.array((0.2470, 0.2435, 0.2616), np.float32)
def preprocess(images):
    x = images.astype(np.float32) / 255.0
    return ((x - MEAN) / STD).transpose(0, 3, 1, 2).copy()

ev = np.load(fetch("cifar_eval.npz"))
X_ev, y_ev = preprocess(ev["images"]), ev["labels"]

class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride=stride, padding=1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, stride=1, padding=1, bias=False); self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = (nn.Sequential(nn.Conv2d(cin, cout, 1, stride=stride, bias=False), nn.BatchNorm2d(cout))
                         if stride != 1 or cin != cout else nn.Identity())
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))

class ResNet(nn.Module):
    def __init__(self, n=3, n_cls=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU(inplace=True))
        self.layer1 = nn.Sequential(*[BasicBlock(16, 16) for _ in range(n)])
        self.layer2 = nn.Sequential(BasicBlock(16, 32, 2), *[BasicBlock(32, 32) for _ in range(n - 1)])
        self.layer3 = nn.Sequential(BasicBlock(32, 64, 2), *[BasicBlock(64, 64) for _ in range(n - 1)])
        self.pool = nn.AdaptiveAvgPool2d(1); self.fc = nn.Linear(64, n_cls)
    def forward(self, x):
        x = self.layer3(self.layer2(self.layer1(self.stem(x))))
        return self.fc(self.pool(x).flatten(1))

def acc_torch(model, X=X_ev, y=y_ev, bs=500):
    model.eval(); dev = next(model.parameters()).device; preds = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            preds.append(model(torch.from_numpy(X[i:i + bs]).to(dev)).argmax(1).cpu().numpy())
    return float((np.concatenate(preds) == y).mean())

rn20 = ResNet(); rn20.load_state_dict(torch.load(fetch("resnet20_cifar10.pt"), map_location="cpu")); rn20.eval()
EX_IN = torch.randn(1, 3, 32, 32)       # torch-pruning 이 연결 구조를 추적할 때 쓰는 예시 입력 (모양만 중요)
FP32_ACC = acc_torch(rn20)
print(f"평가용 {X_ev.shape} | ResNet-20 FP32 정확도 {FP32_ACC:.4f}")

In [ ]:
# ▶ 셀 3 — 경로 튜플로 shortcut 경로 자르기
PATH_ROOTS = [lambda m: m.stem[0],             # 경로 A: stem 의 출력
              lambda m: m.layer2[0].conv2,     # 경로 B: layer2 첫 블록의 출력
              lambda m: m.layer3[0].conv2]     # 경로 C: layer3 첫 블록의 출력

def prune_paths(model, paths, ratio=0.5):
    m = copy.deepcopy(model).cpu().eval()                                   # ① 원본은 그대로 두고 복사본을 자른다
    DG = tp.DependencyGraph().build_dependency(m, example_inputs=EX_IN)    # ② 층 사이 의존 관계 그래프
    imp = tp.importance.MagnitudeImportance(p=1)                           # ③ 중요도 = L1-norm
    for on, get_root in zip(paths, PATH_ROOTS):                            # ④ 경로 하나씩
        if not on:
            continue                                                       #    0 이면 건너뛰기
        root = get_root(m)                                                 # ⑤ 경로의 대표 Conv
        g_all = DG.get_pruning_group(root, tp.prune_conv_out_channels,
                                     idxs=list(range(root.out_channels)))  # ⑥ 이 경로에 묶인 층 전체
        scores = imp(g_all)                                                # ⑦ 채널마다 중요도
        n_cut = int(round(root.out_channels * ratio))                      # ⑧ 자를 개수
        idxs = scores.argsort()[:n_cut].tolist()                           # ⑨ 중요도가 낮은 채널 번호
        DG.get_pruning_group(root, tp.prune_conv_out_channels, idxs=idxs).prune()   # ⑩ 묶음째 자르기
    return m                                                               # ⑪

# 모양 확인 — 1 로 지정한 경로만 줄어야 한다
for d in [(1, 0, 0), (0, 1, 0), (0, 0, 1), (1, 1, 1)]:
    p = prune_paths(rn20, d)
    print(d, "| A", p.stem[0].out_channels, "· B", p.layer2[0].conv2.out_channels,
          "· C", p.layer3[0].conv2.out_channels, "· fc 입력", p.fc.in_features)

In [ ]:
# ▶ 셀 4 — 실험 1 (누적) · 실험 2 (단독)
def run(paths):
    m = rn20 if paths == (0, 0, 0) else prune_paths(rn20, paths)
    macs, params = tp.utils.count_ops_and_params(m, EX_IN)
    return {"paths": str(paths), "macs_m": macs / 1e6, "params": int(params), "acc": acc_torch(m)}

# 실험 1: 누적 — 경로를 하나씩 추가로 자를 때마다 직전 대비 얼마나 더 떨어지나
cum_df = pd.DataFrame([run(d) for d in [(0, 0, 0), (1, 0, 0), (1, 1, 0), (1, 1, 1)]])
cum_df["drop_from_prev"] = cum_df.acc.shift(1) - cum_df.acc

# 실험 2: 단독 — 경로 하나만 자르면 원본 대비 얼마나 떨어지나
solo_df = pd.DataFrame([run(d) for d in [(1, 0, 0), (0, 1, 0), (0, 0, 1)]])
solo_df["drop_from_base"] = FP32_ACC - solo_df.acc          # 비교 대상은 항상 원본

print("실험 1 — 누적"); display(cum_df.round(4))
print("실험 2 — 단독"); display(solo_df.round(4))

In [ ]:
# ▶ 셀 5 (선택) — 단독 실험: 정확도 하락 vs 줄어든 계산량
base_macs = cum_df.macs_m[0]
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(solo_df.paths, solo_df.drop_from_base * 100, color="tab:red"); ax[0].set_title("accuracy drop (%p)")
ax[1].bar(solo_df.paths, (1 - solo_df.macs_m / base_macs) * 100, color="tab:blue"); ax[1].set_title("MACs saved (%)")
plt.tight_layout(); plt.show()

In [ ]:
# ▶ 셀 6 (선택) — 경로별 비율 스윕: 어디까지 버티나
rows = []
for k, name in enumerate(["A", "B", "C"]):
    for r in [0.1, 0.2, 0.3, 0.4, 0.5]:
        paths = tuple(1 if j == k else 0 for j in range(3))
        rows.append({"path": name, "ratio": r, "acc": acc_torch(prune_paths(rn20, paths, ratio=r))})
sweep_df = pd.DataFrame(rows).pivot(index="ratio", columns="path", values="acc")
display(sweep_df.round(4))

## Part 1 결과 읽는 법

| 표 | 열 | 값이 크면 |
|---|---|---|
| 실험 1 (누적) | `drop_from_prev` | 그 경로를 **추가로** 잘랐을 때 크게 떨어졌다 |
| 실험 2 (단독) | `drop_from_base` | 그 경로 **하나만** 잘라도 많이 떨어진다 = 민감한 경로 |
| 공통 | `macs_m`, `params` | 연산량 · 파라미터가 얼마나 남았나 |

단, 여기서의 민감도는 **회복 학습 전** 기준입니다. 회복 학습 후에도 같은 순서가 유지되는지는 Part 2 에서 확인합니다.

---
# Part 2 — 스테이지별 비율 프루닝 + fine-tuning · KD 회복

Part 1 은 shortcut 경로만 잘랐습니다. Part 2 는 **스테이지(A·B·C)마다 비율 하나**를 정하고,
그 스테이지의 shortcut 경로와 블록 내부(conv1 출력 채널)를 **같은 비율로** 자릅니다.
이렇게 해야 전 층을 균등하게 자르는 기준선과 같은 범위를 대상으로 비교할 수 있습니다.

`ratios = (rA, rB, rC)` — 예: `(0.2, 0.3, 0.4)` = A 스테이지 20%, B 30%, C 40%.

**기준선(baseline)**: torch-pruning `MetaPruner` 로 전 층 균등 20% 프루닝 (L1 크기 기준, fc 제외).

**비교 조건**

- 0.1~0.5 다섯 단계 × 3 스테이지 = 125 조합 중, 기준선보다 MACs **와** params 가 **모두** 작은 조합만 후보 (83개)
- 그중 대표 11개 선정: MACs 가 비슷한 묶음(tier) 안에서 자르는 위치(shape)만 바꿔 "같은 연산량이면 어디를 자르는 게 나은가" 비교
- 모든 조합에 같은 학습 데이터 · 에폭 · 학습률 · 평가 세트 · seed 사용
- 기준선 2개(MetaPruner 20%, 스테이지 균등 20%)도 같은 절차로 학습

| 셀 | 내용 | 시간 (T4 기준, 추정) |
|---|---|---|
| 7 | 학습 데이터 · `finetune` · `train_kd` · 기준선 함수 | 수십 초 |
| 8 | `prune_stages` + 125 조합 필터링 | 1분 안팎 |
| 9 | 대표 조합 목록 (수정 가능) | 바로 |
| 10 | 조합마다 프루닝 → FT → KD (CSV 저장, 중단 후 이어서 실행 가능) | 조합당 수십 초 |
| 11 | 이번 run 결과 표 | 바로 |
| 12 | run 평균 → `results/summary.csv`, 그래프 → `figures/` | 바로 |

GPU 가 있으면 자동으로 사용하고, 없으면 CPU 로 실행합니다 (CPU 는 훨씬 느리며, 셀 10 이 남은 시간을 출력합니다).

**반복 실행**: 셀 10 의 `RUN_ID` 를 1, 2, … 로 바꿔 여러 번 실행합니다. run 마다 seed 가 달라서
결과가 학습의 우연에 얼마나 흔들리는지 확인할 수 있습니다.

In [ ]:
# ▶ 셀 7 — 학습 데이터 10,000장 + 학습 함수 (fine-tuning · KD)
import cv2

tr = np.load(fetch("cifar_train10k.npz"))
_m = cv2.imdecode(tr["images_png"], cv2.IMREAD_COLOR)[..., ::-1]          # PNG 모자이크 한 장 → RGB
Xtr_u8 = np.ascontiguousarray(_m.reshape(100, 32, 100, 32, 3).transpose(0, 2, 1, 3, 4).reshape(-1, 32, 32, 3))
ytr = tr["labels"]
print("학습용", Xtr_u8.shape, ytr.shape)

def batches(bs=128, seed=0):
    # 섞기 + 좌우 뒤집기 + 4픽셀 패딩 후 랜덤 자르기 (CIFAR 표준 증강)
    rng = np.random.default_rng(seed); idx = rng.permutation(len(Xtr_u8))
    for i in range(0, len(idx), bs):
        b = Xtr_u8[idx[i:i + bs]]
        b = np.where(rng.random(len(b))[:, None, None, None] < 0.5, b[:, :, ::-1], b)
        pad = np.pad(b, ((0, 0), (4, 4), (4, 4), (0, 0)), mode="reflect"); dx, dy = rng.integers(0, 9, 2)
        yield torch.from_numpy(preprocess(pad[:, dy:dy + 32, dx:dx + 32])).to(DEV), torch.from_numpy(ytr[idx[i:i + bs]]).to(DEV)

STEPS_PER_EPOCH = len(Xtr_u8) // 128 + 1          # 10000/128 → 78 묶음 + 마지막 자투리 1 = 79

def finetune(model, epochs=3, lr=1e-2, seed=0):
    model.to(DEV)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * STEPS_PER_EPOCH)
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        for xb, yb in batches(seed=seed * 1000 + ep):    # run 마다 다른 섞기 순서
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    model.eval()
    return time.time() - t0

def soft(logits, T=1.0):
    return F.softmax(logits / T, dim=1)

def kd_loss(s_logits, t_logits, y, T=4.0, alpha=0.5):
    ce = F.cross_entropy(s_logits, y)                                     # 정답 맞히기
    kl = F.kl_div(F.log_softmax(s_logits / T, dim=1), soft(t_logits, T), reduction="batchmean")  # teacher 분포 따라가기
    return alpha * ce + (1 - alpha) * T**2 * kl

def train_kd(student, teacher, epochs=3, lr=1e-2, T=4.0, alpha=0.5, seed=0):
    student.to(DEV); teacher = teacher.to(DEV).eval()                      # teacher 는 학습하지 않음 → eval 고정
    opt = torch.optim.SGD(student.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * STEPS_PER_EPOCH)
    t0 = time.time()
    for ep in range(epochs):
        student.train()
        for xb, yb in batches(seed=seed * 1000 + ep):    # run 마다 다른 섞기 순서
            with torch.no_grad():
                t_out = teacher(xb)
            loss = kd_loss(student(xb), t_out, yb, T, alpha)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    student.eval()
    return time.time() - t0

def prune_channels(model, ratio):
    # 기준선: MetaPruner 로 전 층을 균등하게 ratio 만큼 (fc 제외)
    m = copy.deepcopy(model).cpu().eval()
    tp.pruner.MetaPruner(m, EX_IN, importance=tp.importance.MagnitudeImportance(p=1),
                         pruning_ratio=ratio, ignored_layers=[m.fc]).step()
    return m

In [ ]:
# ▶ 셀 8 — 스테이지마다 비율 하나: shortcut 경로 + 블록 내부를 함께 자르기
STAGES = [  # (shortcut 경로의 대표 conv, 그 스테이지의 블록들)
    (lambda m: m.stem[0],          lambda m: list(m.layer1)),   # A
    (lambda m: m.layer2[0].conv2,  lambda m: list(m.layer2)),   # B
    (lambda m: m.layer3[0].conv2,  lambda m: list(m.layer3)),   # C
]

def prune_stages(model, ratios):
    m = copy.deepcopy(model).cpu().eval()
    DG = tp.DependencyGraph().build_dependency(m, example_inputs=EX_IN)
    imp = tp.importance.MagnitudeImportance(p=1)

    def cut(root, r):
        n_cut = int(round(root.out_channels * r))
        if n_cut == 0: return
        g = DG.get_pruning_group(root, tp.prune_conv_out_channels, idxs=list(range(root.out_channels)))
        idxs = imp(g).argsort()[:n_cut].tolist()                            # 점수 낮은 채널부터
        DG.get_pruning_group(root, tp.prune_conv_out_channels, idxs=idxs).prune()

    for r, (get_path, get_blocks) in zip(ratios, STAGES):
        if r <= 0: continue
        cut(get_path(m), r)                         # 1) shortcut 경로
        for blk in get_blocks(m): cut(blk.conv1, r) # 2) 블록 내부 (conv1 출력 = conv2 입력)
    return m

def count(m):
    macs, params = tp.utils.count_ops_and_params(m, EX_IN)
    return macs / 1e6, int(params)

BASE_MACS, BASE_PARAMS = count(prune_channels(rn20, 0.2))
print(f"기준선 (균등 20%): MACs {BASE_MACS:.2f}M | params {BASE_PARAMS:,}")

# 125 조합의 크기만 계산 (정확도는 측정하지 않으므로 빠름)
import itertools
LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5]
grid = []
for rs in itertools.product(LEVELS, repeat=3):
    macs, params = count(prune_stages(rn20, rs))
    grid.append({"ratios": rs, "macs_m": macs, "params": params,
                 "pass": macs <= BASE_MACS and params <= BASE_PARAMS})
grid_df = pd.DataFrame(grid)
print(f"기준선보다 MACs·params 둘 다 작은 조합: {grid_df['pass'].sum()} / {len(grid_df)}")

In [ ]:
# ▶ 셀 9 — 대표 조합 (tier = MACs 비슷한 묶음, shape = 어디를 많이 자르나)
#   uniform(균등): 모두 같게 / increasing(증가): 뒤(C)로 갈수록 많이 / decreasing(감소): 앞(A)을 많이 / middle(가운데): B 를 많이
REPS = [
    ("~20M", "uniform",    (0.3, 0.3, 0.3)),
    ("~20M", "increasing", (0.2, 0.3, 0.4)),
    ("~20M", "decreasing", (0.4, 0.3, 0.2)),
    ("~20M", "middle",     (0.2, 0.5, 0.2)),
    ("~15M", "uniform",    (0.4, 0.4, 0.4)),
    ("~15M", "increasing", (0.3, 0.4, 0.5)),
    ("~15M", "decreasing", (0.5, 0.4, 0.3)),
    ("~15M", "middle",     (0.3, 0.5, 0.3)),
    ("~11M", "uniform",    (0.5, 0.5, 0.5)),
    ("~11M", "increasing", (0.4, 0.5, 0.5)),
    ("~11M", "decreasing", (0.5, 0.5, 0.4)),
]
REFS = [  # 기준선 (필터 통과 여부와 무관하게 실행)
    ("baseline", "MetaPruner 20%", None),
    ("baseline", "stage 20%",      (0.2, 0.2, 0.2)),
]

lookup = grid_df.set_index("ratios")
for tier, shape, rs in REPS:
    row = lookup.loc[[rs]].iloc[0]
    print(f"{tier:5s} {shape:10s} {rs}  MACs {row.macs_m:6.2f}M  params {row.params:7,}  {'통과' if row['pass'] else '✗ 기준선보다 큼'}")

In [ ]:
# ▶ 셀 10 — 실험: 자르기 → (그대로 / fine-tuning / KD) 정확도
#   조합 하나가 끝날 때마다 CSV 에 저장 → 중단 후 다시 실행하면 완료된 조합은 건너뜀
EPOCHS = 3
RUN_ID = 1             # ← run 번호 (1, 2, … 로 바꿔 반복 실행)
SEED = RUN_ID - 1      # run 1 → seed 0, run 2 → seed 1
OUT = os.path.join(RESULTS_DIR, f"run{RUN_ID}.csv")

done = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame()
done_keys = set(done["name"]) if len(done) else set()
teacher = copy.deepcopy(rn20)          # 원본 rn20 은 그대로 두고 teacher 는 복사본 사용
jobs = REFS + REPS
t_start = time.time(); n_new = 0

for i, (tier, shape, rs) in enumerate(jobs):
    name = f"{tier}/{shape}"
    if name in done_keys:
        print(f"[{i+1}/{len(jobs)}] {name} — 이미 있음, 건너뜀"); continue

    pruned = prune_channels(rn20, 0.2) if rs is None else prune_stages(rn20, rs)
    macs, params = count(pruned)
    acc0 = acc_torch(pruned)

    torch.manual_seed(SEED); ft = copy.deepcopy(pruned); t_ft = finetune(ft, EPOCHS, seed=SEED); acc_ft = acc_torch(ft)
    torch.manual_seed(SEED); kd = copy.deepcopy(pruned); t_kd = train_kd(kd, teacher, EPOCHS, seed=SEED); acc_kd = acc_torch(kd)

    row = {"run": RUN_ID, "seed": SEED, "name": name, "tier": tier, "shape": shape, "ratios": str(rs), "macs_m": round(macs, 2), "params": params,
           "acc0": acc0, "acc_ft": acc_ft, "acc_kd": acc_kd, "t_ft": round(t_ft, 1), "t_kd": round(t_kd, 1)}
    done = pd.concat([done, pd.DataFrame([row])], ignore_index=True); done.to_csv(OUT, index=False)

    n_new += 1; left = sum(1 for t, s, _ in jobs[i+1:] if f"{t}/{s}" not in done_keys)
    eta = (time.time() - t_start) / n_new * left
    print(f"[{i+1}/{len(jobs)}] {name:28s} MACs {macs:5.2f}M | 그대로 {acc0:.3f} → FT {acc_ft:.3f} / KD {acc_kd:.3f}"
          f" | {t_ft + t_kd:.0f}초 | 남은 시간 약 {eta/60:.1f}분")

print("저장:", OUT)

In [ ]:
# ▶ 셀 11 — 이번 run 결과 보기
res = pd.read_csv(OUT)
res["kd_gain"] = res.acc_kd - res.acc_ft                       # KD 가 FT 보다 더 회복한 정도
res["gap_to_fp32"] = FP32_ACC - res[["acc_ft", "acc_kd"]].max(axis=1)
display(res[["tier", "shape", "ratios", "macs_m", "params", "acc0", "acc_ft", "acc_kd", "kd_gain", "gap_to_fp32"]].round(4))

reps = res[res.tier != "baseline"]
print("\nKD 정확도 — tier × shape"); display(reps.pivot(index="shape", columns="tier", values="acc_kd").round(4))

In [ ]:
# ▶ 셀 12 — run 들을 합쳐 평균 내기 → results/summary.csv, 그림 → figures/
#   run*.csv 는 원본 기록이므로 수정하지 않고, 평균은 별도 파일로 저장
import glob
runs = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(os.path.join(RESULTS_DIR, "run*.csv")))], ignore_index=True)
print("합친 run:", sorted(runs.run.unique()))

keys = ["name", "tier", "shape", "ratios", "macs_m", "params"]          # run 과 무관하게 같은 값 (프루닝 결과는 결정적)
summary = (runs.groupby(keys, dropna=False, sort=False)
               .agg(n_runs=("run", "nunique"),
                    acc0=("acc0", "mean"),
                    acc_ft=("acc_ft", "mean"), acc_ft_std=("acc_ft", "std"),
                    acc_kd=("acc_kd", "mean"), acc_kd_std=("acc_kd", "std"))
               .reset_index())
summary["kd_gain"] = summary.acc_kd - summary.acc_ft
summary["gap_to_fp32"] = FP32_ACC - summary.acc_kd
summary.round(4).to_csv(os.path.join(RESULTS_DIR, "summary.csv"), index=False)
display(summary.round(4))

# 그래프 라벨 (기준선은 앞에 "baseline:" 을 붙임)
summary["label"] = np.where(summary.tier == "baseline", "baseline: " + summary["shape"], summary["shape"])
COLOR = {"uniform": "tab:gray", "increasing": "tab:blue", "decreasing": "tab:red", "middle": "tab:green"}

def pareto(df, x):
    # x 가 작은 순으로 보며 지금까지보다 정확도가 높은 점만 남김 = 더 나은 대안이 없는 점(Pareto front)
    best, keep = -1, []
    for i, r in df.sort_values(x).iterrows():
        if r.acc_kd > best: keep.append(i); best = r.acc_kd
    return df.loc[keep]

def plot(x, xlabel, fname):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    reps = summary[summary.tier != "baseline"]
    for lab, g in reps.groupby("label"):
        ax.scatter(g[x], g.acc_kd, color=COLOR[lab], label=lab, s=50)
        ax.scatter(g[x], g.acc_ft, color=COLOR[lab], marker="x", alpha=0.4)     # x 표시 = 같은 조합의 FT
    for _, r in summary[summary.tier == "baseline"].iterrows():
        ax.scatter(r[x], r.acc_kd, marker="*", s=220, color="k")
        ax.annotate(r.label, (r[x], r.acc_kd), textcoords="offset points", xytext=(6, -12), fontsize=8)
    p = pareto(reps, x); ax.plot(p[x], p.acc_kd, "k--", lw=1, alpha=0.6, label="Pareto front (KD)")
    ax.axhline(FP32_ACC, ls=":", color="gray", label=f"unpruned {FP32_ACC:.3f}")
    ax.set_xlabel(xlabel); ax.set_ylabel("accuracy (KD: dot, FT: x)")
    ax.set_title(f"{xlabel} vs accuracy  (mean of {summary.n_runs.max()} runs)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout()
    path = os.path.join(FIG_DIR, fname); fig.savefig(path, dpi=150, bbox_inches="tight"); plt.show()
    print("저장:", path)

plot("macs_m", "MACs (M)", "pareto_macs.png")
plot("params", "params", "pareto_params.png")

## Part 2 결과 읽는 법

| 열 | 의미 |
|---|---|
| `acc0` | 프루닝 직후, 회복 학습 전 정확도 |
| `acc_ft` / `acc_kd` | fine-tuning / KD 로 회복한 정확도 |
| `kd_gain` | KD − FT. 양수면 teacher(원본 모델)의 출력 분포를 따라 학습한 것이 도움이 됨 |
| `gap_to_fp32` | 원본 모델 정확도까지 남은 차이 |
| `*_std` | run 사이 표준편차 (학습 무작위성에 따른 흔들림) |

- **같은 tier 안에서** shape 를 비교 → 같은 연산량이면 어디를 자르는 것이 유리한가
- **★ 기준선**보다 왼쪽(연산량 적음)이면서 비슷하거나 높은 위치의 점 → 균등 프루닝보다 효율적인 조합
- tier 가 내려갈수록(~20M → ~11M) 회복 정도 → 어디까지 줄일 수 있는가

**해석 시 주의**: 평가 세트가 2,000장이라 정확도 1장 = 0.05%p 이고, 0.8 부근 표준오차는 약 ±0.9%p 입니다.
run 반복은 학습 무작위성만 줄이며 평가 세트 크기에서 오는 오차는 줄이지 못합니다.

### 확장 아이디어

- `EPOCHS` 를 늘려 KD 의 이점이 유지되는지 확인
- 셀 9 에 조합 추가 — `grid_df[grid_df["pass"]]` 에서 선택
- KD 하이퍼파라미터 변경 → `train_kd(kd, teacher, EPOCHS, T=2.0, alpha=0.3)`